# Google Trends — weekly update (frozen params)

`main.ipynb` does a full refit: decides which keywords need clustering, then fits the
denoise lambda and detrend trend from scratch on the whole history. That's the expensive,
occasional-use pipeline — rerun it every so often (e.g. once a season) to refresh the
clustering decision and the fitted params, and to re-baseline this notebook's inputs.

This notebook is the cheap weekly path: it treats the clustering decision and the fitted
denoise/detrend params as frozen (like a gridsearch's tuned hyperparameters — found once,
reused many times) and just **replays** them (`apply_denoise_test` / `apply_detrend_test`)
onto only the newly-arrived weeks, appending to the existing preprocessed file. No
clustering recomputation, no lambda grid search, no ADF tests.

**Download scope note:** the Health Trends API normalizes each response relative to the
queried date range, so re-querying a short recent window and appending it to the existing
raw history would create a scale discontinuity at the boundary. To stay safe, the download
step below still requests the full `start_date` → today range (same API-call count as
before, just a bigger response) — what's actually skipped is redeciding *which*
keywords/clusters to query and refitting denoise/detrend, not shrinking the query window.

**Why detrend can't just replay over everything:** `apply_detrend_test` continues each
series' fitted trend from exactly where the original fit's training data left off (it
indexes new rows starting at `train_len`). Feeding it the *entire* re-downloaded history
(instead of only the rows newer than the last full fit) would misindex the trend and
produce garbage for any series with an active linear/quadratic detrend — this notebook
explicitly slices to `date > meta["data_max_date"]` before the detrend-replay step to
avoid that.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PARAMS_DIR = PROCESSED_DIR / "params"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import json
import pandas as pd
from lgbm_forecast import gtrends_download, io, gtrends_denoise, gtrends_detrend

print("import OK")


## Step 1 — Load the frozen artifacts

Saved by `main.ipynb`'s "Save fit artifacts" cell the last time it was run in full:
- `ok_all.json` — which keywords are usable standalone, per location (the OK/CLUSTER decision)
- `cluster_all.csv` — the combined query terms per location for the CLUSTER-flagged keywords
- `denoise_summary.json` — fitted lambda per (location, variable)
- `detrend_summary.json` — fitted trend (action/coef/mean_ref/train_len) per (location, variable)
- `meta.json` — the date the params were fit through (`data_max_date`); rows newer than this
  are the ones this notebook actually needs to produce

If any of these are missing, run `main.ipynb` in full first.

In [ ]:
with open(PARAMS_DIR / "ok_all.json") as f:
    ok_all = json.load(f)

cluster_terms = pd.read_csv(PROCESSED_DIR / "cluster_all.csv")

denoise_summary = pd.read_json(PARAMS_DIR / "denoise_summary.json")
detrend_summary = pd.read_json(PARAMS_DIR / "detrend_summary.json")

with open(PARAMS_DIR / "meta.json") as f:
    meta = json.load(f)

LOCATIONS = meta["locations"]
DATA_MAX_DATE = pd.to_datetime(meta["data_max_date"])

print(f"params fit on {meta['fit_date']}, data through {DATA_MAX_DATE.date()}")
print(f"{len(LOCATIONS)} locations, {sum(len(v) for v in ok_all.values())} OK keyword series, "
      f"{len(cluster_terms)} clustered-location series")
print(f"denoise_summary: {len(denoise_summary)} series, detrend_summary: {len(detrend_summary)} series")


## Step 2 — Download (full range, only the already-decided keywords/clusters)

In [ ]:
key = ''


In [ ]:
RUN_DOWNLOAD = False

download_df = pd.concat([
    pd.DataFrame([{"location": loc, "cluster_id": 0, "terms": label} for label in cols])
    for loc, cols in ok_all.items()
] + [cluster_terms], ignore_index=True)

if RUN_DOWNLOAD:
    api_key = key
    gtrends_download.download_all(
        cluster_df=download_df,
        save_dir=str(RAW_DIR),
        api_key=api_key,
        start_date="2014-01-01",
        end_date=pd.Timestamp.today().strftime("%Y-%m-%d"),
    )

print(f"{len(download_df)} series to (re)download")
download_df.head()


## Step 3 — Reload raw series into long format

In [ ]:
google_df = io.load_timeseries_wide(str(RAW_DIR), LOCATIONS)
google_df = google_df.drop(columns=["cough cold & flu"], errors="ignore")

google_cluster = io.load_timeseries_wide(str(RAW_DIR), LOCATIONS, only_clusters=True)

# Same cheap 30%-zero recheck main.ipynb does -- not a fitted param, just a fresh
# data check, so it's fine (and more correct) to recompute against current data.
cluster_zero_check = (
    google_cluster.groupby("location")["cluster_all"]
    .apply(lambda s: (s == 0).mean() * 100)
    .reset_index(name="zero_pct")
)
cluster_zero_check["under_30pct"] = cluster_zero_check["zero_pct"] < 30
location_no_cluster = cluster_zero_check[cluster_zero_check["under_30pct"] == False]["location"].tolist()
print(f"Locations dropped (cluster still >=30% zero): {location_no_cluster}")

long_ok = gtrends_denoise.make_selected_long(google_df, ok_all)
cluster_dict = {loc: ["cluster_all"] for loc in google_cluster["location"].unique()}
for loc in location_no_cluster:
    cluster_dict.pop(loc, None)
long_cluster = gtrends_denoise.make_selected_long(google_cluster, cluster_dict)

long_all = pd.concat([long_ok, long_cluster], ignore_index=True)
long_all["date"] = pd.to_datetime(long_all["date"])
print(f"{long_all.shape[0]} rows, date range {long_all['date'].min()} to {long_all['date'].max()}")

n_new_weeks = long_all.loc[long_all['date'] > DATA_MAX_DATE, 'date'].nunique()
print(f"{n_new_weeks} new week(s) since the params were fit ({DATA_MAX_DATE.date()})")


## Step 4 — Denoise (replay, no refit)

`apply_denoise_test` reuses each series' fitted lambda from `denoise_summary` instead of
re-running the grid search. Run over the *full* re-downloaded history (not just the new
rows) since the rolling smoother needs the trailing 20-week window of raw context to
produce a value at each new date -- this is cheap (no fitting), so there's no reason to
bother slicing it down.

In [ ]:
denoised_all = gtrends_denoise.apply_denoise_test(long_all, denoise_summary)
print(denoised_all["denoised_value"].isna().sum(), "NaN denoised values")
denoised_all.head()


## Step 5 — Detrend only the new rows (replay, no refit)

Unlike denoise, `apply_detrend_test` **must** only see rows strictly newer than
`DATA_MAX_DATE` -- see the note at the top. Slice first, then replay.

In [ ]:
denoised_all["date"] = pd.to_datetime(denoised_all["date"])
denoised_new = denoised_all[denoised_all["date"] > DATA_MAX_DATE].copy()
denoised_new["date"] = denoised_new["date"].dt.strftime("%Y-%m-%d")

if denoised_new.empty:
    print("No new rows since the last fit -- nothing to detrend/append.")
else:
    detrended_new = gtrends_detrend.apply_detrend_test(denoised_new, detrend_summary)
    print(detrended_new["detrended_value"].isna().sum(), "NaN detrended values")
    print(f"{detrended_new.shape[0]} new rows across "
          f"{pd.to_datetime(detrended_new['date']).min().date()} to "
          f"{pd.to_datetime(detrended_new['date']).max().date()}")
detrended_new.head()


## Step 6 — Stationarity recheck on the new rows (diagnostic only)

In [ ]:
recheck = gtrends_detrend.recheck_stationarity_adf(detrended_new, column="detrended_value")
recheck["stationarity_status"].value_counts()


## Save — append the new rows to the existing preprocessed file

The existing file's historical rows are untouched (they came from the last full fit);
only the newly-replayed rows get pivoted and appended.

In [ ]:
new_wide = detrended_new.pivot_table(
    index=["date", "location"], columns="variable", values="detrended_value", aggfunc="first"
).reset_index()
new_wide.columns.name = None
new_wide["date"] = pd.to_datetime(new_wide["date"])

out_path = PROCESSED_DIR / "google_trends_preprocessed.csv"
existing = pd.read_csv(out_path, parse_dates=["date"])

combined = pd.concat([existing, new_wide], ignore_index=True)
combined = combined.drop_duplicates(subset=["date", "location"], keep="last")
combined = combined.sort_values(["location", "date"]).reset_index(drop=True)

combined.to_csv(out_path, index=False)
print(f"Saved: {out_path}  {existing.shape} -> {combined.shape}  max date: {combined['date'].max().date()}")
combined.tail(5)


## Refresh the frozen params for next time

Update `meta.json` so the next run of this notebook knows the new cutoff, and re-save the
denoise/detrend summaries -- they didn't change (still the params from the last full
`main.ipynb` run), this just keeps `data_max_date` in sync so Step 5 slices correctly next
time.

In [ ]:
meta["data_max_date"] = str(combined["date"].max().date())
with open(PARAMS_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Updated meta.json data_max_date ->", meta["data_max_date"])
